# 02: Clean Transactions Table
Standardize column names and data types, combine date and time into a single timestamp, assign a transaction ID, and validate the clean table against the raw source.

In [2]:
import duckdb
import pandas as pd
from pathlib import Path

DB_PATH = Path("../data/processed/aml.duckdb")
con = duckdb.connect(str(DB_PATH))

con.sql("SHOW TABLES").df()

,name
0,transactions
1,transactions_raw


In [3]:
con.execute("""
    CREATE OR REPLACE TABLE transactions AS
    SELECT
        ROW_NUMBER() OVER (
            ORDER BY Date, Time, Sender_account, Receiver_account, Amount
        )                                    AS txn_id,
        CAST(Date + Time AS TIMESTAMP)       AS txn_ts,
        Date                                 AS txn_date,
        CAST(Sender_account AS VARCHAR)      AS sender_account,
        CAST(Receiver_account AS VARCHAR)    AS receiver_account,
        CAST(Amount AS DECIMAL(18, 2))       AS amount,
        Payment_currency                     AS payment_currency,
        Received_currency                    AS received_currency,
        Sender_bank_location                 AS sender_location,
        Receiver_bank_location               AS receiver_location,
        Payment_type                         AS payment_type,
        CAST(Is_laundering AS TINYINT)       AS is_laundering,
        Laundering_type                      AS laundering_type
    FROM transactions_raw
""")

con.sql("SELECT * FROM transactions LIMIT 5").df()

,txn_id,txn_ts,txn_date,sender_account,receiver_account,amount,payment_currency,received_currency,sender_location,receiver_location,payment_type,is_laundering,laundering_type
0,1,2022-10-07 10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits
1,2,2022-10-07 10:35:20,2022-10-07,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque,0,Normal_Small_Fan_Out
2,3,2022-10-07 10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0,Normal_Fan_Out
3,4,2022-10-07 10:35:21,2022-10-07,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_In
4,5,2022-10-07 10:35:21,2022-10-07,8974559268,3143547511,5130.99,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Group


## Validation: Clean vs. Raw
Reconcile row counts, total amount, and laundering labels, then check nulls, value ranges, and duplicates.

In [4]:
con.sql("""
    SELECT
        'raw'                                         AS source,
        COUNT(*)                                      AS n_rows,
        CAST(ROUND(SUM(Amount), 2) AS DECIMAL(20, 2)) AS total_amount,
        CAST(SUM(Is_laundering) AS BIGINT)            AS n_laundering
    FROM transactions_raw

    UNION ALL

    SELECT
        'clean',
        COUNT(*),
        SUM(amount),
        CAST(SUM(is_laundering) AS BIGINT)
    FROM transactions
""").df()

,source,n_rows,total_amount,n_laundering
0,raw,9504852,8.329071e+10,9873
1,clean,9504852,8.329071e+10,9873


In [5]:
con.sql("""
    SELECT
        COUNT(*) - COUNT(txn_ts)           AS null_ts,
        COUNT(*) - COUNT(sender_account)   AS null_sender,
        COUNT(*) - COUNT(receiver_account) AS null_receiver,
        COUNT(*) - COUNT(amount)           AS null_amount,
        COUNT(*) - COUNT(payment_type)     AS null_payment_type,
        MIN(amount)                        AS min_amount,
        MAX(amount)                        AS max_amount,
        SUM(CASE WHEN amount <= 0 THEN 1 ELSE 0 END) AS n_zero_or_negative,
        MIN(txn_ts)                        AS first_ts,
        MAX(txn_ts)                        AS last_ts
    FROM transactions
""").df()

,null_ts,null_sender,null_receiver,null_amount,null_payment_type,min_amount,max_amount,n_zero_or_negative,first_ts,last_ts
0,0,0,0,0,0,3.73,12618498.4,0.0,2022-10-07 10:35:19,2023-08-23 10:57:12


In [6]:
con.sql("""
    WITH dupes AS (
        SELECT
            txn_ts, sender_account, receiver_account, amount, payment_type,
            COUNT(*) AS n
        FROM transactions
        GROUP BY txn_ts, sender_account, receiver_account, amount, payment_type
        HAVING COUNT(*) > 1
    )
    SELECT
        (SELECT COUNT(*) FROM transactions)                 AS n_rows,
        (SELECT COUNT(DISTINCT txn_id) FROM transactions)   AS n_unique_txn_ids,
        (SELECT COUNT(*) FROM dupes)                        AS n_duplicate_groups,
        (SELECT COALESCE(SUM(n - 1), 0) FROM dupes)         AS n_extra_duplicate_rows
""")

n_rowsBIGINT,n_unique_txn_idsBIGINT,n_duplicate_groupsBIGINT,n_extra_duplicate_rowsHUGEINT
9504852,9504852,0,0


In [7]:
con.sql("""
    SELECT
        (SELECT SUM(amount) FROM transactions)
      - (SELECT CAST(ROUND(SUM(Amount), 2) AS DECIMAL(20, 2)) FROM transactions_raw)
        AS amount_difference
""").df()

,amount_difference
0,0.0


### Validation Result
The clean `transactions` table reconciles fully to the raw source. Row counts (9,504,852), laundering labels (9,873), and total transaction value match between both tables. There are no null values in any key field, no zero or negative amounts, and every transaction has a unique ID with no duplicate records. Amounts range from £3.73 to £12.6 million, which indicates a heavily right-skewed distribution that will need to be considered when setting thresholds by segment. The clean table is ready to serve as the foundation for transaction-code mapping and scenario development.

## Transaction-Code Mapping
Map each raw payment type to a standardized transaction code, product, channel, and cash indicator, then confirm every transaction maps to a code so no activity falls outside monitoring coverage.

In [8]:
con.execute("""
    CREATE OR REPLACE TABLE txn_code_map AS
    SELECT * FROM (VALUES
        ('Cash Deposit',    'CSH-DEP', 'Cash',       'Deposit Account',    'Branch / ATM',     TRUE,  'High'),
        ('Cash Withdrawal', 'CSH-WDL', 'Cash',       'Deposit Account',    'Branch / ATM',     TRUE,  'High'),
        ('Cross-border',    'WIR-INT', 'Wire',       'International Wire', 'SWIFT',            FALSE, 'High'),
        ('Cheque',          'CHQ',     'Paper',      'Deposit Account',    'Cheque Clearing',  FALSE, 'Standard'),
        ('ACH',             'ACH',     'Electronic', 'Deposit Account',    'Electronic Batch', FALSE, 'Standard'),
        ('Debit card',      'DC-POS',  'Card',       'Deposit Account',    'Card Network',     FALSE, 'Standard'),
        ('Credit card',     'CC-POS',  'Card',       'Credit Card',        'Card Network',     FALSE, 'Standard')
    ) AS t(payment_type, txn_code, txn_category, product, channel, is_cash, inherent_risk)
""")

con.sql("SELECT * FROM txn_code_map").df()

,payment_type,txn_code,txn_category,product,channel,is_cash,inherent_risk
0,Cash Deposit,CSH-DEP,Cash,Deposit Account,Branch / ATM,True,High
1,Cash Withdrawal,CSH-WDL,Cash,Deposit Account,Branch / ATM,True,High
2,Cross-border,WIR-INT,Wire,International Wire,SWIFT,False,High
3,Cheque,CHQ,Paper,Deposit Account,Cheque Clearing,False,Standard
4,ACH,ACH,Electronic,Deposit Account,Electronic Batch,False,Standard
5,Debit card,DC-POS,Card,Deposit Account,Card Network,False,Standard
6,Credit card,CC-POS,Card,Credit Card,Card Network,False,Standard


In [9]:
con.sql("""
    SELECT
        t.payment_type,
        m.txn_code,
        m.inherent_risk,
        COUNT(*) AS n_txns
    FROM transactions AS t
    LEFT JOIN txn_code_map AS m
        ON t.payment_type = m.payment_type
    GROUP BY t.payment_type, m.txn_code, m.inherent_risk
    ORDER BY n_txns DESC
""").df()

,payment_type,txn_code,inherent_risk,n_txns
0,Credit card,CC-POS,Standard,2012909
1,Debit card,DC-POS,Standard,2012103
2,Cheque,CHQ,Standard,2011419
3,ACH,ACH,Standard,2008807
4,Cross-border,WIR-INT,High,933931
5,Cash Withdrawal,CSH-WDL,High,300477
6,Cash Deposit,CSH-DEP,High,225206


In [10]:
con.sql("""
    SELECT
        payment_type,
        COUNT(*) AS n_txns,
        SUM(CASE WHEN sender_location <> receiver_location   THEN 1 ELSE 0 END) AS n_geo_cross_border,
        SUM(CASE WHEN payment_currency <> received_currency  THEN 1 ELSE 0 END) AS n_currency_mismatch
    FROM transactions
    GROUP BY payment_type
    ORDER BY n_txns DESC
""").df()

,payment_type,n_txns,n_geo_cross_border,n_currency_mismatch
0,Credit card,2012909,0.0,37838.0
1,Debit card,2012103,0.0,37972.0
2,Cheque,2011419,0.0,37714.0
3,ACH,2008807,0.0,37737.0
4,Cross-border,933931,933931.0,929753.0
5,Cash Withdrawal,300477,412.0,412.0
6,Cash Deposit,225206,5.0,4436.0


In [11]:
con.execute("""
    COPY txn_code_map
    TO '../docs/txn_code_mapping.csv'
    (HEADER, DELIMITER ',')
""")
print("Saved to docs/txn_code_mapping.csv")

Saved to docs/txn_code_mapping.csv


### Mapping Impact Analysis
The payment-type label alone does not fully capture cross-border or foreign currency risk. All Cross-border payments involve different sender and receiver locations, but 417 cash transactions (412 withdrawals and 5 deposits) also cross borders, likely representing cash accessed abroad. A scenario that relies only on the Cross-border payment type would miss this activity. Separately, roughly 150,000 domestic card, cheque, and ACH payments involve a currency mismatch, which shows that cross-border movement and foreign currency exposure are distinct risk attributes. To close these gaps, cross-border and foreign currency indicators will be derived directly from location and currency fields rather than from the payment-type label.

In [12]:
con.execute("""
    CREATE OR REPLACE TABLE transactions_enriched AS
    SELECT
        t.*,
        m.txn_code,
        m.txn_category,
        m.product,
        m.channel,
        m.is_cash,
        m.inherent_risk,
        (t.sender_location  <> t.receiver_location)  AS is_cross_border,
        (t.payment_currency <> t.received_currency)  AS is_foreign_currency
    FROM transactions AS t
    LEFT JOIN txn_code_map AS m
        ON t.payment_type = m.payment_type
""")

con.sql("SELECT * FROM transactions_enriched LIMIT 5").df()

,txn_id,txn_ts,txn_date,sender_account,receiver_account,amount,payment_currency,received_currency,sender_location,receiver_location,...,is_laundering,laundering_type,txn_code,txn_category,product,channel,is_cash,inherent_risk,is_cross_border,is_foreign_currency
0,1,2022-10-07 10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,...,0,Normal_Cash_Deposits,CSH-DEP,Cash,Deposit Account,Branch / ATM,True,High,False,False
1,2,2022-10-07 10:35:20,2022-10-07,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,...,0,Normal_Small_Fan_Out,CHQ,Paper,Deposit Account,Cheque Clearing,False,Standard,False,False
2,3,2022-10-07 10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,...,0,Normal_Fan_Out,WIR-INT,Wire,International Wire,SWIFT,False,High,True,True
3,4,2022-10-07 10:35:21,2022-10-07,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,...,0,Normal_Fan_In,ACH,Electronic,Deposit Account,Electronic Batch,False,Standard,False,False
4,5,2022-10-07 10:35:21,2022-10-07,8974559268,3143547511,5130.99,UK pounds,UK pounds,UK,UK,...,0,Normal_Group,ACH,Electronic,Deposit Account,Electronic Batch,False,Standard,False,False


In [13]:
con.sql("""
    SELECT
        COUNT(*)                                                AS n_rows,
        SUM(CASE WHEN txn_code IS NULL      THEN 1 ELSE 0 END)  AS n_unmapped,
        SUM(CASE WHEN is_cross_border       THEN 1 ELSE 0 END)  AS n_cross_border,
        SUM(CASE WHEN is_foreign_currency   THEN 1 ELSE 0 END)  AS n_foreign_currency
    FROM transactions_enriched
""").df()

,n_rows,n_unmapped,n_cross_border,n_foreign_currency
0,9504852,0.0,934348.0,1085862.0


In [14]:
con.sql("""
    SELECT
        is_cross_border,
        is_foreign_currency,
        COUNT(*)                                AS n_txns,
        CAST(SUM(is_laundering) AS BIGINT)      AS n_laundering,
        ROUND(AVG(is_laundering) * 100, 3)      AS pct_laundering
    FROM transactions_enriched
    GROUP BY is_cross_border, is_foreign_currency
    ORDER BY pct_laundering DESC
""").df()

,is_cross_border,is_foreign_currency,n_txns,n_laundering,pct_laundering
0,True,False,4178,39,0.933
1,False,True,155692,651,0.418
2,True,True,930170,3006,0.323
3,False,False,8414812,6177,0.073


### Derived Flag Risk Analysis
Both derived indicators carry meaningful risk signal. Cross-border transactions without a currency conversion show the highest laundering rate at 0.93%, roughly 13 times the 0.07% baseline for domestic, same-currency activity. Domestic transactions involving a foreign currency launder at 0.42%, about 6 times the baseline, and contain 651 laundering transactions that a scenario keyed only on the Cross-border payment type would never review. At the same time, about 63% of all laundering occurs in domestic, same-currency activity, which confirms that geographic and currency indicators alone are not sufficient and that behavioral scenarios such as structuring and rapid movement of funds remain essential.

In [15]:
con.close()
print("Connection closed.")

Connection closed.
